# M8 — Version C (smart search) + the bigger-catalog test

**Two questions this step answers**
1. **Do "advanced RAG" tricks make search better?** Version C = B + three tricks, each tested on and off.
2. **What happens when the catalog grows from 28 to 100 to 250 tables?** A real company has hundreds of tables.
   With 28, "show the AI everything" (A) won. Does that still hold at 250?

**Version C's three tricks**
| Trick | What it does | Analogy |
|---|---|---|
| **Rewrite** | the AI first rewrites the question as a description of the table it needs | asking a librarian to translate your question into catalog terms |
| **Hybrid search** | search by **meaning** (dense) *and* by **keywords** (BM25), then merge (Reciprocal Rank Fusion) | looking in the subject index *and* the word index |
| **Re-rank** | a cross-encoder reads the question and each of the top 20 cards **together** and re-scores them | a careful second reader who re-orders the shortlist |

An **ablation test** switches one trick off at a time. If accuracy drops when a trick is off, that trick helps.

**The bigger catalog:** extra tables from public benchmarks, **TPC-DS** (24 retail tables), **TPC-H** (8) and
**BIRD** (real databases from a text-to-SQL benchmark), are added to the *catalog only* (cards, search index,
table map). They're never part of a correct answer; they're noise, like the many unrelated tables in a real
warehouse. Their cards were drafted by the AI in the same style as ours, so writing style doesn't give them away.
Catalog 100 = our 28 + 72 extra; catalog 250 = our 28 + 222 extra.

**Files in this step**
| File | What it does |
|---|---|
| `catalog/distractors.py` | reads TPC-DS / TPC-H / BIRD schemas, drafts cards, `load_catalog(28/100/250)` |
| `catalog/distractor_cards/` | the 222 extra cards + `ORDER.txt` (fixed order, seed 42) |
| `catalog/load_qdrant.py` | now stores a meaning vector **and** a keyword vector per card, for all 3 catalog sizes |
| `catalog/smart_search.py` | version C's search: rewrite, hybrid, re-rank (each switchable) |
| `catalog/load_neo4j.py` | now also puts the extra tables (and their joins) in the map |
| `agent/pipelines.py` | adds `version_c()`; every version can use a bigger catalog |
| `harness/run.py` | new switch `--catalog 100` / `--catalog 250` |
| `configs/C*.yaml` | C with all tricks, and with each trick switched off |

**Done when:** the ablation and bigger-catalog results are saved ✅

In [1]:
import sys, json, glob
for folder in ["../agent", "../catalog", "../harness", "../mcp_server"]:
    sys.path.append(folder)

## 1. The extra tables

In [2]:
from distractors import load_catalog
from collections import Counter
big = load_catalog(250)
extra = [c for c in big.values() if c.get("source")]
print(len(big), "tables in the 250 catalog;", len(extra), "extra:", dict(Counter(c["source"] for c in extra)))
for name in ["tpcds_promotion", "tpcds_web_sales"] + [c["table"] for c in extra if c["source"] == "BIRD"][:3]:
    print(f"\n{name}: {big[name]['description']}")

250 tables in the 250 catalog; 222 extra: {'TPCDS': 24, 'TPCH': 8, 'BIRD': 190}

tpcds_promotion: Details of marketing promotions (IDs, start/end dates, linked item, cost, response target, channels, purpose, and discount status) used to analyze and join promotion information in sales and marketing reports.

tpcds_web_sales: Detailed web sales line items for TPC-DS: one record per item sold online including sold/ship dates and times, item, customer billing/shipping demographics and addresses, pricing, discounts, taxes, shipping costs, promotions, warehouse and net profit — used for revenue, margin and customer/order analytics.

superstore_east_superstore: Order line items for the East region of the Superstore dataset: each row contains order, shipping, customer, product and financial details (sales, quantity, discount, profit) used for sales, fulfillment and performance analysis.

disney_characters: A table of Disney movies listing each movie's title, release date, main hero, main villa

Some extra tables are deliberately "nearby" (TPC-DS has promotions, customers, web sales, dates), which makes
the search harder. Others (BIRD) are unrelated, like most tables in a big company.

## 2. Smart search, one trick at a time
The same tricky question, with each trick on or off. The right table is `agg_campaign_daily_valid`.

In [3]:
from smart_search import smart_search
q = "What was campaign 143747's CTR yesterday, excluding suspicious clicks?"
for rewrite, hybrid, rerank in [(False, False, False), (True, False, False), (False, True, False), (False, False, True), (True, True, True)]:
    tables, _ = smart_search(q, use_rewrite=rewrite, use_hybrid=hybrid, use_rerank=rerank)
    label = "B (none)" if not (rewrite or hybrid or rerank) else " + ".join(n for n, on in [("rewrite", rewrite), ("hybrid", hybrid), ("rerank", rerank)] if on)
    print(f"{label:28s} -> {tables[:3]}")

/Users/anshikapuneet/Documents/Claude_history/capstone_project/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


B (none)                     -> ['agg_campaign_hourly_valid', 'agg_campaign_daily_valid', 'agg_ad_daily_valid']


rewrite                      -> ['agg_campaign_daily_valid', 'agg_ad_daily_valid', 'agg_campaign_hourly_valid']
hybrid                       -> ['agg_campaign_hourly_valid', 'agg_campaign_daily_valid', 'agg_campaign_daily']


rerank                       -> ['agg_campaign_daily_valid', 'agg_ad_daily_valid', 'agg_campaign_hourly_valid']


rewrite + hybrid + rerank    -> ['agg_campaign_daily_valid', 'agg_ad_daily_valid', 'agg_campaign_hourly_valid']


## 3. Ablation results (28 tables, 40 practice questions)

In [4]:
def latest(name, suffix="practice"):
    folders = sorted(glob.glob(f"../results/*_{name}_{suffix}"))
    return json.load(open(folders[-1] + "/metrics.json")) if folders else {}

rows = {"B (no tricks)": latest("B"), "C (all three)": latest("C"),
        "C without rewrite": latest("C_no_rewrite"), "C without hybrid": latest("C_no_hybrid"),
        "C without re-rank": latest("C_no_rerank")}
keys = ["accuracy", "accuracy_medium", "accuracy_tricky", "gold_tables_shown", "avg_seconds", "avg_cost_usd"]
print(f"{'':20s}" + "".join(f"{k[:16]:>18s}" for k in keys))
for name, m in rows.items():
    print(f"{name:20s}" + "".join(f"{m.get(k, '-'):>18}" for k in keys))

                              accuracy   accuracy_medium   accuracy_tricky  gold_tables_show       avg_seconds      avg_cost_usd
B (no tricks)                      0.7            0.3077            0.7857              0.65            4.2538            0.0007
C (all three)                    0.675            0.2308            0.7857              0.65             9.965            0.0013
C without rewrite                0.775            0.4615            0.8571             0.725             5.981            0.0007
C without hybrid                 0.675            0.3077            0.7857             0.625           11.0587            0.0013
C without re-rank                0.775            0.5385            0.7857               0.7             8.527            0.0012


## 4. The bigger-catalog test: 28 → 100 → 250 tables
```
.venv/bin/python harness/run.py configs/A.yaml --split practice --catalog 250
```

In [5]:
sizes = [28, 100, 250]
table = {}
for v in ["A", "B", "C", "D"]:
    table[v] = {size: latest(v, "practice" if size == 28 else f"practice_cat{size}") for size in sizes}

for metric in ["accuracy", "gold_tables_shown", "avg_tokens_in", "avg_cost_usd"]:
    print(f"\n{metric}")
    print(f"{'version':10s}" + "".join(f"{s:>12}" for s in sizes))
    for v in table:
        print(f"{v:10s}" + "".join(f"{table[v][s].get(metric, '-'):>12}" for s in sizes))


accuracy
version             28         100         250
A                0.875         0.9         0.9
B                  0.7         0.7         0.7
C                0.675         0.7         0.7
D                 0.85        0.85         0.8

gold_tables_shown
version             28         100         250
A                  1.0         1.0         1.0
B                 0.65        0.65        0.65
C                 0.65       0.625       0.575
D                 0.85       0.875       0.875

avg_tokens_in
version             28         100         250
A               3208.8     12958.8     29022.8
B              712.825       720.2       720.2
C                872.5     885.275      881.95
D              911.225     917.925     917.925

avg_cost_usd
version             28         100         250
A               0.0012      0.0037      0.0077
B               0.0007      0.0007      0.0007
C               0.0013      0.0013      0.0013
D               0.0007      0.0007      0.0007


## 5. What we found (practice questions: 40 only, so a difference of 1–2 questions is noise)

**Smart search (C) did not beat basic search (B).** C 67.5% vs B 70%.
- **Rewrite hurt.** Switching it off *raised* accuracy to 77.5%. It also costs an extra AI call, which doubles the cost and adds about 5 s.
- **Re-rank hurt too.** Switching it off gave 77.5%.
- **Hybrid helped a little.** Switching it off gave 67.5%, the same as or lower than with it.
- **Why:** C usually *does* find the main table (`agg_ad_daily`) now, but many medium questions also need a small
  **lookup table** to join with (`dim_ad` for brand names, `dim_user` for gender/age). Search ranks tables one at
  a time, so it doesn't know that "you'll also need `dim_ad`". The table map (D) does know, because it follows
  `JOINS_WITH`. **That is why D (85%) beats every C variant.** Better search can't fix a joining problem.

**Bigger catalog (28 → 100 → 250 tables)**
- **A ("show everything") stayed accurate** (87.5% → 90% → 90%), but what it reads grew from about 3k to about 29k tokens per question,
  so its **cost went up about 6×** ($0.0012 → $0.0077). At a few thousand tables it wouldn't fit at all.
- **B, C and D cost the same at any size** (they always read 5–8 cards).
- **D dipped a little at 250** (85% → 80%) and C's `gold_tables_shown` fell (65% → 57.5%): the extra tables start
  to crowd the search results.
- **Honest caveat:** most extra tables (BIRD: movies, superstores, law episodes...) are about very different topics,
  so they rarely look like ad tables. This test shows **cost** growing more clearly than **confusion**. A catalog
  of hundreds of *ad* tables would be a harder test.

We didn't tune C after seeing these results. They're reported as they came out.

**Next step (M9):** final runs on the 80 **test** questions, 3 runs each, with confidence intervals, plus the
failure analysis.